Importando e tratando os dados considerando a cidade de São Paulo

Henrique: considerações que eu fiz no processamento
  - Se a região não tem classificação (i.e., NaN), ipvs é setado como 0
  - Estou excluindo as estações da CETESB que estão na RMSP, mas não estão na cidade de São Paulo (Osasco, Guarulhos, etc)

In [1]:
import pandas as pd
import geopandas as gpd

from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
# Configurações

## UBSs
considerar_nulos_internet = True # Há várias células vazias na coluna de conexão com internet. Se True, considera que elas têm internet.

# 1. Processamento do IPVS

In [3]:
ipvs = gpd.read_file("raw_data/ipvs_2022/IPVS_2022.shp")

# Seleção de valores e drops de colunas
ipvs = ipvs[(ipvs["NM_MUN"] == "São Paulo")] # Filtra apenas os setores da cidade de São Paulo
ipvs.loc[ipvs["C_IPVS"].isna(), "C_IPVS"] = 0 # Substitui os valores NaN por 0 na coluna C_IPVS
ipvs.drop(columns=["fid", "CD_MUN", "NM_MUN"], inplace=True)
ipvs = ipvs.reset_index(drop=True)

# O sistema de coordenadas do shapefile é diferente do sistema de coordenadas geográficas (latitude e longitude)
# Então precisamos converter as coordenadas para o sistema geográfico para obter as latitudes e longitudes corretas.

# epsg = numero que o claude falou pra usar pra sp EPSG:31983 (SIRGAS 2000 / UTM zone 23S), 
ipvs_proj = ipvs.to_crs(epsg=31983)
# espg 31983 é de metros, nos queremos em latitude e longitude
centroides = ipvs_proj.geometry.centroid.to_crs(epsg=4326)

# Adicionar as colunas de latitude e longitude ao DataFrame
ipvs["latitude"] = centroides.y # Y = Latitude
ipvs["longitude"] = centroides.x # X = Longitude

print(ipvs.shape)       # quantas linhas e colunas tem
print(ipvs.head())      # primeiras 5 linhas

(27301, 9)
          CD_SETOR SITUACAO    CD_DIST      NM_DIST                 N_IPVS  \
0  355030811000549   Urbana  355030811  Brasilândia  Média Vulnerabilidade   
1  355030811000548   Urbana  355030811  Brasilândia  Média Vulnerabilidade   
2  355030811000558   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   
3  355030811000557   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   
4  355030811000556   Urbana  355030811  Brasilândia   Alta Vulnerabilidade   

   C_IPVS                                           geometry   latitude  \
0     4.0  POLYGON ((-46.69624 -23.456, -46.69629 -23.456... -23.456297   
1     4.0  POLYGON ((-46.69781 -23.45723, -46.69775 -23.4... -23.456620   
2     5.0  POLYGON ((-46.71246 -23.45803, -46.71238 -23.4... -23.458155   
3     5.0  POLYGON ((-46.69942 -23.44628, -46.69949 -23.4... -23.445496   
4     5.0  POLYGON ((-46.69998 -23.44609, -46.69949 -23.4... -23.447601   

   longitude  
0 -46.697527  
1 -46.696698  
2 -46.711812  
3 -46.698

### 1.5 Adicionar dados de populacao aos dados do IPVS (IBGE CENSO 2022)

In [4]:
censo = pd.read_csv("raw_data/dados_censo/Agregados_por_setores_basico_BR_20250417.csv", sep=";", encoding="iso-8859-1", dtype={"CD_SETOR": str, "CD_MUN": str}, low_memory=False, quotechar='"')

# Filtrar somente municipio de Sao Paulo (codigo IBGE 3550308)
censo_sp = censo[censo["CD_MUN"] == "3550308"][["CD_SETOR", "v0001"]].copy()
censo_sp = censo_sp.rename(columns={"v0001": "POPULACAO"})

# v0001 vem com virgula decimal no arquivo; normalizamos para numerico
censo_sp["POPULACAO"] = pd.to_numeric(censo_sp["POPULACAO"].astype(str).str.replace(",", ".", regex=False), errors="coerce")

# Merge com o IPVS
ipvs = ipvs.merge(censo_sp, on="CD_SETOR", how="left")

print(ipvs[["CD_SETOR", "C_IPVS", "POPULACAO"]].head(10))
print("Nulos em população:", ipvs["POPULACAO"].isna().sum())

          CD_SETOR  C_IPVS  POPULACAO
0  355030811000549     4.0        447
1  355030811000548     4.0        418
2  355030811000558     5.0        501
3  355030811000557     5.0        315
4  355030811000556     5.0        455
5  355030811000555     4.0        372
6  355030811000554     5.0        385
7  355030811000552     2.0        287
8  355030811000559     5.0        430
9  355030811000561     5.0        405
Nulos em população: 0


# 2. Processamento das UBSs

In [5]:
# Extrair dados do Cadastro Nacional de Estabelecimentos de Saúde (CNES)
cnes = pd.read_csv('raw_data/dados_cnes/tbEstabelecimento202602.csv', sep=';' ,encoding='latin-1', low_memory=False)

# Filtrar apenas as UBSs da cidade de São Paulo
ubs = cnes[(cnes["TP_UNIDADE"] == 2) & (cnes["CO_MUNICIPIO_GESTOR"] == 355030) & (cnes["NO_RAZAO_SOCIAL"] == "PREFEITURA DO MUNICIPIO DE SAO PAULO") & (cnes["CO_MOTIVO_DESAB"].isna()) & (cnes["TP_GESTAO"] == "M")] # Filtra apenas as linhas onde TP_UNIDADE é 2 (UBS), CO_MUNICIPIO_GESTOR é 355030 (São Paulo), NO_RAZAO_SOCIAL é "PREFEITURA DO MUNICIPIO DE SAO PAULO", CO_MOTIVO_DESAB é NaN (não desativada), e TP_GESTAO é "M" (municipal)

if considerar_nulos_internet:
    ubs = ubs[ubs["ST_CONEXAO_INTERNET"] != "N"] # Filtra apenas as linhas onde ST_CONEXAO_INTERNET não é "N"
else:
    ubs = ubs[ubs["ST_CONEXAO_INTERNET"] == "S"] # Filtra apenas as linhas onde ST_CONEXAO_INTERNET é "S"

# Drops de colunas
ubs.drop(columns=["NU_CNPJ_MANTENEDORA", "TP_PFPJ", "NIVEL_DEP", "NO_RAZAO_SOCIAL", "NO_LOGRADOURO", "NU_ENDERECO", "NO_COMPLEMENTO", "CO_CEP", "CO_REGIAO_SAUDE", "CO_MICRO_REGIAO", "CO_DISTRITO_SANITARIO", "CO_DISTRITO_ADMINISTRATIVO", "NU_TELEFONE", "NU_FAX", "NO_EMAIL", "NU_CPF", "NU_CNPJ", "CO_ATIVIDADE", "CO_CLIENTELA", "NU_ALVARA", "DT_EXPEDICAO", "TP_ORGAO_EXPEDIDOR", "DT_VAL_LIC_SANI", "TP_LIC_SANI", "TP_UNIDADE", "CO_TURNO_ATENDIMENTO", "CO_ESTADO_GESTOR", "CO_MUNICIPIO_GESTOR", "TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')", "CO_USUARIO", "CO_CPFDIRETORCLN", "REG_DIRETORCLN", "ST_ADESAO_FILANTROP", "CO_MOTIVO_DESAB", "NO_URL", "TO_CHAR(DT_ATU_GEO,'DD/MM/YYYY')", "NO_USUARIO_GEO", "CO_NATUREZA_JUR", "TP_ESTAB_SEMPRE_ABERTO", "ST_GERACREDITO_GERENTE_SGIF", "ST_CONEXAO_INTERNET", "CO_TIPO_UNIDADE", "NO_FANTASIA_ABREV", "TP_GESTAO", "TO_CHAR(DT_ATUALIZACAO_ORIGEM,'DD/MM/YYYY')", "CO_TIPO_ESTABELECIMENTO", "CO_ATIVIDADE_PRINCIPAL", "CO_TIPO_ABRANGENCIA", "ST_COWORKING", "ST_CONTRATO_FORMALIZADO"], inplace=True)
ubs = ubs.reset_index(drop=True)

print(ubs.columns.tolist())
print(ubs.head(3))

['CO_UNIDADE', 'CO_CNES', 'NO_FANTASIA', 'NO_BAIRRO', 'NU_LATITUDE', 'NU_LONGITUDE']
      CO_UNIDADE  CO_CNES                                        NO_FANTASIA  \
0  3550302064820  2064820  UBS PONTE RASA DR CARLOS OLIVALDO DE SOUZA LOP...   
1  3550302788381  2788381            UBS PARQUE REGINA PERINA ALVES TEIXEIRA   
2  3550302787369  2787369                                  UBS JARDIM ELIANE   

            NO_BAIRRO          NU_LATITUDE         NU_LONGITUDE  
0  ALTO DA PONTE RASA  -23.514951583709312  -46.483072946798224  
1       PARQUE REGINA           -23.634587           -46.755694  
2       JARDIM ELIANA          -23.7535296          -46.6716892  


# 3. Processamento das estações da CETESB

In [6]:
# Extrair dados das estações de monitoramento da qualidade do ar da CETESB
cetesb = pd.read_excel("raw_data/postos-de-cetesb.xlsx")

# Filtrar apenas as estações da cidade de São Paulo
cetesb = cetesb[(cetesb["Município"] == "São Paulo")]

cetesb.drop(columns=["UF", "Município", "COD", "Observações"], inplace=True) # não lembro oq significava Automático x Manual, tirar dps se necessário
cetesb = cetesb.replace({"NÃO": 0, "SIM": 1})

cetesb = cetesb.reset_index(drop=True)

# As coordenadas da CETESB estão em UTM 23S (SIRGAS 2000); convertemos para graus decimais (WGS84)
cetesb = gpd.GeoDataFrame(
    cetesb,
    geometry=gpd.points_from_xy(
        pd.to_numeric(cetesb["LONGITUDE"], errors="coerce"),
        pd.to_numeric(cetesb["LATITUDE"], errors="coerce")
    ),
    crs="EPSG:31983"
)

cetesb = cetesb.to_crs(epsg=4326)
cetesb["LATITUDE"] = cetesb.geometry.y
cetesb["LONGITUDE"] = cetesb.geometry.x
cetesb.drop(columns="geometry", inplace=True)
    
print(cetesb.head())
print(cetesb.columns.tolist())

                   Nome Estacao        TIPO   LATITUDE  LONGITUDE Acetaldeido  \
0                Campos Elíseos      Manual -23.533190 -46.644603           0   
1                 Capão Redondo  Automática -23.668356 -46.780043           0   
2               Cerqueira César  Automática -23.553543 -46.672705           0   
3               Cerqueira César      Manual -23.553543 -46.672705           0   
4  Cidade Universitária – USP –  Automática -23.566342 -46.737414           0   

  Benzeno CO Enxofre Reduzido Total FMC Formaldeido MP10 MP2,5 NO NO2 NOx O3  \
0       0  0                      0   1           0    0     0  0   0   0  0   
1       0  0                      0   0           0    1     0  0   0   0  1   
2       0  1                      0   0           0    1     0  1   1   1  0   
3       0  0                      0   1           0    0     1  0   0   0  0   
4       0  0                      0   0           0    0     1  0   0   0  1   

  PTS SO2 Tolueno  
0   0   1   

# Salvar variáveis no disco

In [7]:
output_dir = Path("processed_data")
output_dir.mkdir(parents=True, exist_ok=True)

tabelas_processadas = {
    "ipvs": ipvs,
    "ubs": ubs,
    "cetesb": cetesb
}

for nome_tabela, df in tabelas_processadas.items():
    caminho_saida = output_dir / f"{nome_tabela}.parquet"
    df.to_parquet(caminho_saida, index=False)

print(f"Tabelas salvas em: {output_dir.resolve()}")
print("Arquivos gerados:")
for arquivo in sorted(output_dir.glob("*.parquet")):
    print(f" - {arquivo.name}")

Tabelas salvas em: /home/ubuntu/pontos-instalacao/pollution-sensing-optimization/data/processed_data
Arquivos gerados:
 - cetesb.parquet
 - ipvs.parquet
 - ubs.parquet
